#### Importing the library

In [1]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
#Printing stopwords in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

#### Data Collection and Data preprocessing

In [3]:
#Load the dataset
news_dataset = pd.read_csv('news_dataset.csv')

In [4]:
news_dataset.shape

(6000, 5)

In [5]:
# print the first 5 data
news_dataset.head()

,title,text,subject,date,label
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1


In [6]:
#Counting the missing values in the dataset
news_dataset.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [7]:
#Merging the News Title and the News Subject (Im not using this merging because of it just seems wrong)
news_dataset['content'] = news_dataset['title']

In [8]:
print(news_dataset['content'])

0        WATCH: Trump Just Told All The Anti-Gay Bigot...
1       China backs U.N. call for justice in Yemen, U....
2       THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...
3       WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...
4       Lawmakers aim to delay U.S. ceding control of ...
                              ...                        
5995    Exiled Venezuelan opposition magistrates resur...
5996          Plague outbreak in Madagascar kills 20: WHO
5997    Trump to visit Asia in November, North Korea i...
5998    Melania Trump calls taped comments by Donald T...
5999    #BoycottPenzeys: HATEFUL, DIVISIVE Penzeys Spi...
Name: content, Length: 6000, dtype: object


In [9]:
#Separating the data label
x = news_dataset.drop(columns='label', axis=1)
y = news_dataset['label']

In [10]:
print(x)
print(y)

                                                  title  \
0      WATCH: Trump Just Told All The Anti-Gay Bigot...   
1     China backs U.N. call for justice in Yemen, U....   
2     THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...   
3     WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...   
4     Lawmakers aim to delay U.S. ceding control of ...   
...                                                 ...   
5995  Exiled Venezuelan opposition magistrates resur...   
5996        Plague outbreak in Madagascar kills 20: WHO   
5997  Trump to visit Asia in November, North Korea i...   
5998  Melania Trump calls taped comments by Donald T...   
5999  #BoycottPenzeys: HATEFUL, DIVISIVE Penzeys Spi...   

                                                   text       subject  \
0     A whole lot of evangelical Trump voters just d...          News   
1     GENEVA (Reuters) - China signaled on Wednesday...     worldnews   
2                                                            politics   

#### Stemming
Reducing the word to its root word

In [11]:
port_stem = PorterStemmer()

In [12]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [13]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [14]:
print(news_dataset['content'])

0       watch trump told anti gay bigot mike penc go f ck
1                china back u n call justic yemen u saudi
2       peopl presid trump meet coal worker fulfil cam...
3       wsj report rip dem candid lame rhetor isi rema...
4          lawmak aim delay u cede control internet manag
                              ...                        
5995       exil venezuelan opposit magistr resurfac chile
5996                       plagu outbreak madagascar kill
5997        trump visit asia novemb north korea spotlight
5998    melania trump call tape comment donald trump b...
5999    boycottpenzey hate divis penzey spice co owner...
Name: content, Length: 6000, dtype: object


In [15]:
#Separating the data label
x = news_dataset['content'].values
y = news_dataset['label'].values

In [16]:
print(x)

['watch trump told anti gay bigot mike penc go f ck'
 'china back u n call justic yemen u saudi'
 'peopl presid trump meet coal worker fulfil campaign promis video' ...
 'trump visit asia novemb north korea spotlight'
 'melania trump call tape comment donald trump boy talk'
 'boycottpenzey hate divis penzey spice co owner threaten trump support brag increas sale sinc call trump support racist']


In [17]:
print(y)

[0 1 0 ... 1 1 0]


In [18]:
y.shape

(6000,)

In [19]:
#Converting textual data to numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(x)

x = vectorizer.transform(x)

In [20]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 54560 stored elements and shape (6000, 6772)>
  Coords	Values
  (0, 240)	0.2651895903974571
  (0, 572)	0.39933982124861395
  (0, 1068)	0.37396619388535823
  (0, 2448)	0.34775555713056794
  (0, 2513)	0.27953384796705505
  (0, 3779)	0.3646886817339425
  (0, 4375)	0.3457181068658953
  (0, 6093)	0.33825991933158767
  (0, 6204)	0.11601705452739947
  (0, 6551)	0.2255655126547208
  (1, 395)	0.3745163930599012
  (1, 847)	0.32471310422550387
  (1, 1030)	0.3710113422544801
  (1, 3214)	0.45021977582753797
  (1, 5201)	0.4144738910976388
  (1, 6727)	0.4921006017516659
  (2, 864)	0.2892912359448619
  (2, 1129)	0.4162490706849936
  (2, 2401)	0.473916870316174
  (2, 3717)	0.2750268192820156
  (2, 4388)	0.3003593854578868
  (2, 4605)	0.23616945527968666
  (2, 4659)	0.3590543565288969
  (2, 6204)	0.12250805869935917
  (2, 6464)	0.1548233906550441
  :	:
  (5997, 6486)	0.3414165119591157
  (5998, 700)	0.40263218482195234
  (5998, 847)	0.2656772

#### Splitting the dataset into training and test

In [21]:
x_train, x_test, y_train, y_test =  train_test_split(x, y, test_size = 0.2, stratify = y, random_state = 2)

#### Training the model: Logistic Regression

In [22]:
model = LogisticRegression()

Training the model on train dataset

In [23]:
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


#### Evaluation
Accuracy score

In [24]:
# Accuracy score on the training data
x_train_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(x_train_prediction, y_train)

In [25]:
print('Accuracy score on the training data:', training_data_accuracy)

Accuracy score on the training data: 0.9645833333333333


In [26]:
x_test_prediction = model.predict(x_test)
test_data_accuracy = accuracy_score(x_test_prediction, y_test)

In [27]:
print('Accuracy score on the test data:', test_data_accuracy)

Accuracy score on the test data: 0.9033333333333333


#### Making a predictive system

In [45]:
x_new = x_test[142]

prediction = model.predict(x_new)
print(prediction)

if prediction[0] == 0:
    print('The news is real')
else:
    print('The news is fake')

print("The real label:",y_test[142])

[1]
The news is fake
The real label: 1
